# The model fits the data it was shown. Does it look like the data?

A fitted model reproduces the outcome's mean almost by construction. What it need not
reproduce is the outcome's *spread*, its extremes, or its stickiness from one period to the
next — and a model that gets those wrong will forecast badly, size experiments wrongly, and
produce intervals that are confidently the wrong width, all while fitting beautifully in
sample.

The posterior predictive check is the version of "look at the residuals" that has a number
attached: simulate replicate outcomes from the posterior, and ask where the real data's
statistics fall among them.

After a fit: does the model reproduce the observed outcome's summary statistics, and do the
residuals look like the noise the likelihood assumed? `posterior_predictive` simulates replicate
outcomes from posterior draws and compares each `Statistic` of the data against its predictive
interval and tail probability; `residuals` runs Durbin–Watson, Ljung–Box, normality, and
Breusch–Pagan tests on the posterior-mean residuals, each with its statistic, p-value, and the
`N` it used.

In [ ]:
import numpy as np

from axiom.core import Prior, Unsupported
from axiom.diagnose import (
    DEFAULT_STATISTICS, PPCResult, ResidualReport, ResidualTest, Statistic, StatisticCheck,
    UnitResiduals, posterior_predictive, residuals,
)
from axiom.sim import DosePlan, surface_world
from axiom.surface import HillKernel, fit

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import CRITICAL, caption, compare, intervals, shade

enable();  # every axiom result renders itself from here on

AMP = Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 0.5})
world = surface_world(n_units=4, n_periods=12, treatments=("a",), kernels=HillKernel(reference_dose=1.0, amplitude_prior=AMP), intercept="shared", noise_sd=0.3, seed=7, doses=DosePlan(zero_fraction=0.2))
res = fit(world.spec, world.panel, backend="laplace", draws=100, chains=1, seed=0)
print("converged:", res.converged, "| draws:", res.n_draws())

## `posterior_predictive`

`DEFAULT_STATISTICS` maps names to `Statistic` callables (mean, sd, min, max, lag-1
autocorrelation, ...). Each `StatisticCheck` carries the observed value, the predictive `eti`
interval with its mass, the one- and two-sided tail probabilities over `n` replicates, and
`extreme` at the stated alpha. A statistic that returns a non-finite value is listed in
`skipped` rather than silently dropped.

In [ ]:
print("default statistics:", list(DEFAULT_STATISTICS))
ppc = posterior_predictive(res, n_draws=80, seed=1)
assert isinstance(ppc, PPCResult)
rows = []
for s in ppc.statistics:
    assert isinstance(s, StatisticCheck)
    rows.append([s.name, f"{s.observed:.3f}",
                 f"[{s.interval.lower:.3f}, {s.interval.upper:.3f}]",
                 f"{s.p_two_sided:.3f}", str(s.extreme)])
table(rows, headers=("statistic", "observed", "predictive interval", "two-sided p", "extreme"))
print("extreme:", ppc.extreme_statistics, "| round-trips:", PPCResult.from_json(ppc.to_json()) == ppc)


def spread(y: np.ndarray) -> float:
    return float(np.max(y) - np.min(y))


custom: dict[str, Statistic] = {"spread": spread, "nan": lambda y: float("nan")}
ppc2 = posterior_predictive(res, statistics=custom, n_draws=40, seed=2)
assert isinstance(ppc2, PPCResult)
print("custom:", [s.name for s in ppc2.statistics], "| skipped:", ppc2.skipped)

In [ ]:
placements = []
for check in ppc.statistics:
    lo, hi = check.interval.lower, check.interval.upper
    width = (hi - lo) or 1.0
    placements.append((f"{check.name}  (p = {check.p_two_sided:.2f})",
                       (check.observed - lo) / width, 0.0, 1.0))
fig = intervals(
    placements, ref=0.5, ref_label="centre",
    highlight=next((row[0] for row, chk in zip(placements, ppc.statistics) if chk.extreme), None),
    title="Where the real data falls among the model's replicates",
    subtitle=f"each statistic's {ppc.statistics[0].interval.mass:.0%} predictive interval, rescaled to [0, 1]",
    x_title="position of the observed value within its predictive interval",
)
caption(fig, "Every bar is one statistic's predictive interval, stretched to a common scale so "
             "a mean in outcome units and a lag-1 autocorrelation can be read side by side. A "
             "dot at 0 or 1 is a statistic the model cannot reproduce; a dot outside the bar "
             "would be one it never produced in eighty tries.")

## `residuals`

Per-unit `UnitResiduals` (mean and sd over the unit's periods) and a `ResidualTest` per check.
Durbin–Watson has no p-value (its bounds depend on the design) and is reported as a statistic
only; Ljung–Box runs at each requested lag below the number of periods, and the rest are
skipped with a reason. `flagged` lists the tests with `p < alpha`.

In [ ]:
rr = residuals(res, lags=(1, 4, 20), alpha=0.05)
assert isinstance(rr, ResidualReport)
print(f"n={rr.n} ({rr.n_units} units x {rr.n_periods} periods), residual sd={rr.residual_sd:.3f}")
rows = []
for u in rr.units:
    assert isinstance(u, UnitResiduals)
    rows.append([u.unit, f"{u.mean:.3f}", f"{u.sd:.3f}", u.n])
table(rows, headers=("unit", "mean residual", "sd", "n"))
rows = []
for t in rr.tests:
    assert isinstance(t, ResidualTest)
    rows.append([t.name, f"{t.statistic:.3f}",
                 "n/a" if t.p_value is None else f"{t.p_value:.3f}", t.n, t.note])
table(rows, headers=("test", "statistic", "p", "n", "note"))
print("skipped:", rr.skipped, "| flagged:", rr.flagged)

In [ ]:
fig = intervals(
    [(f"unit {u.unit}", u.mean, u.mean - 2 * u.sd / np.sqrt(u.n), u.mean + 2 * u.sd / np.sqrt(u.n))
     for u in rr.units],
    ref=0.0, ref_label="unbiased",
    title="Is the model wrong about one unit in particular?",
    subtitle="mean residual per unit, ±2 standard errors of that mean",
    x_title="mean residual",
)
caption(fig, "A residual mean that misses zero for one unit is a unit the model is "
             "systematically wrong about — the signature of a missing unit-level term, which "
             "an aggregate residual sd hides completely.")
print("round-trips:", ResidualReport.from_json(rr.to_json()) == rr)
declined = fit(world.spec, world.panel, backend="no-such-backend")
print("no posterior ->", type(residuals(declined)).__name__, "/", type(posterior_predictive(declined)).__name__)
assert isinstance(residuals(declined), Unsupported)

## What this bought you

Six statistics of the data checked against what the model would produce, each with a tail
probability and its N; per-unit residual means that expose a systematically mis-fitted unit;
and four residual tests that skip with a reason rather than reporting a p-value they cannot
compute.